![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 01: Foundations)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- Teaching content is licensed under CC BY 4.0 and code under MIT; see [LICENSING.md](../../LICENSING.md) for scope and exclusions.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-ai](https://github.com/tulip-lab/agentic-ai/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 1B: GenAI and Agentic AI Fundamentals

<div align="center">

<table>
<thead>
<tr>
<th><strong>Item</strong></th>
<th><strong>Description</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">Estimated time</td>
<td>2 hours</td>
</tr>
<tr>
<td align="left">Environment</td>
<td>Google Colab or local Jupyter</td>
</tr>
<tr>
<td align="left">Main output</td>
<td>A small rule-based GenAI/agentic AI concept demonstrator using unit-repository text</td>
</tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m01b-overview)
2. [Setup and Background](#m01b-setup)
3. [Core Concepts](#m01b-core-concepts)
4. [Guided Implementation](#m01b-guided-implementation)
5. [Testing and Analysis](#m01b-testing)
6. [Student Tasks](#m01b-student-tasks)
7. [Submission and Reflection](#m01b-submission)

---

<a id="m01b-overview"></a>

### 1. Overview and Learning Goals

This session introduces the conceptual foundation of generative AI and agentic AI. Later sessions in this unit will build chatbots, RAG systems, LangChain agents, LangGraph workflows, multi-agent systems and evaluation pipelines. Before building those systems, you need a clear mental model of what a large language model does, how prompts and context affect output, and how an agentic workflow differs from a simple chatbot.

A simple chatbot usually receives a user message and returns a response. An agentic workflow is more structured. It may inspect the task, choose a tool, retrieve documents, update state, verify output and ask for human confirmation before taking action. This difference matters because agentic systems are more powerful, but they also introduce more failure modes. A chatbot can hallucinate an answer; an agent can hallucinate an answer and then call a tool or modify external state.

This session therefore treats agentic AI as a workflow-design problem rather than only a model-selection problem. A model is only one component. A useful agentic system also needs input boundaries, task decomposition, tool permissions, state representation, retrieval strategy, validation logic and evaluation evidence. These ideas will reappear in [M04B-LangChain-ToolAgents](../../M04-Agent-Programming/Jupyter/M04B-LangChain-ToolAgents.ipynb), [M05C-LangGraph-StatefulWorkflows](../../M05-Knowledge-Agents/Jupyter/M05C-LangGraph-StatefulWorkflows.ipynb) and [M06B-LLM-Malicious-Instruction-Defense](../../M06-Multi-Agent-Safety/Jupyter/M06B-LLM-Malicious-Instruction-Defense.ipynb).

In this notebook, we will not call a live LLM API. Instead, we will build a small conceptual demonstrator using public unit text. This keeps the lab reproducible and avoids requiring an API key in the first conceptual session. The exercise uses public unit-repository material as the input text, consistent with the unit data policy. For later data-driven practicals, simple public datasets should be drawn from [tulip-lab/open-data](https://github.com/tulip-lab/open-data).

By the end of this lab, you should be able to explain the difference between generative AI, LLMs, chatbots and agentic AI; identify the role of prompts, context and parameters; describe why tool use changes the risk profile of an AI system; build a small concept classifier over unit text; interpret the classifier output; and test the classifier with normal, edge and failure cases.

<a id="m01b-setup"></a>

### 2. Setup and Background

This notebook uses only the Python standard library. That is intentional. The goal of this session is to understand concepts before adding external model APIs or frameworks. Later sessions will use Gemini, LangChain, LangGraph, Flowise and Hugging Face. Here, we focus on the core ideas in a lightweight and reproducible way.

The example text below is based on the public FLIP unit description and syllabus themes. When you later build RAG or retrieval-based practicals, you should use public unit documents such as `README.md`, `SYLLABUS.md`, public handouts, or public datasets from [tulip-lab/open-data](https://github.com/tulip-lab/open-data). This keeps the exercises reproducible and avoids privacy or copyright problems.

The following table gives working definitions used in this session. These terms are deliberately practical. They are not meant to replace formal definitions, but they help you understand how later notebooks and Flowise tutorials are organised.

<div align="center">

<table>
<thead>
<tr>
<th><strong>Concept</strong></th>
<th><strong>Meaning in this lab</strong></th>
<th><strong>Where it appears later in the unit</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left">Prompt</td><td>The instruction or question given to an AI system.</td><td>M01C, M03B, M04A</td></tr>
<tr><td align="left">Context</td><td>Additional information provided to help the system answer.</td><td>M02C, M03C, M05A</td></tr>
<tr><td align="left">Tool</td><td>An external function, API or workflow component the system can call.</td><td>M03D, M04B, M04C</td></tr>
<tr><td align="left">State</td><td>Structured memory of what has happened in a workflow.</td><td>M05C, M05D</td></tr>
<tr><td align="left">Evaluation</td><td>A systematic way to check whether the system behaves as expected.</td><td>M07E, M08A</td></tr>
<tr><td align="left">Safety boundary</td><td>A rule that limits what the system is allowed to do.</td><td>M06B, M06D</td></tr>
</tbody>
</table>

</div>

In [ ]:
# This session intentionally uses only the Python standard library.
# Later sessions will introduce LLM APIs, LangChain, LangGraph, Flowise and Hugging Face.

import re
import textwrap
from collections import Counter
from typing import Dict, List, Any

print("Setup complete.")

In [ ]:
# Public unit-repository sample text.
# In a full repository setting, this text can be replaced by content loaded from README.md or SYLLABUS.md
# in this unit repository. Later practicals such as M02C, M03C, M05A and M05B may also use public
# datasets from https://github.com/tulip-lab/open-data.
#
# The text is deliberately short so that students can inspect it directly.
# Later RAG practicals will use a larger set of public unit documents.

unit_text = """
FLIP: Agentic AI in Practice is a practical unit for building generative and agentic AI systems.
The unit uses Python notebooks, visual workflows, retrieval-augmented generation, multi-agent systems,
model adaptation, evaluation and coding-agent practice.

Public unit documents and public datasets should be used wherever possible in exercises.
Simple data exercises should preferably use the tulip-lab open-data repository:
https://github.com/tulip-lab/open-data

The unit covers foundations of generative AI, large language models, prompts, APIs,
function calling, tool use, Flowise workflows, LangChain programming, LangGraph stateful workflows,
RAG over unit materials, multi-agent collaboration, prompt injection defence, private agents with
open-source LLMs, model adaptation, multimodal generation, evaluation and productised AI agents.
"""

print(textwrap.fill(unit_text.strip(), width=100))

The text above acts as a miniature public document corpus. Later practicals such as [M03C-Flowise-RAG-Public-Unit-Docs](../../M03-Context-Orchestration/Flowise/M03C-Flowise-RAG-Public-Unit-Docs.md), [M05A-Basic-RAG-System](../../M05-Knowledge-Agents/Jupyter/M05A-Basic-RAG-System.ipynb) and [M05B-RAG-CourseMaterials-Assistant](../../M05-Knowledge-Agents/Jupyter/M05B-RAG-CourseMaterials-Assistant.ipynb) will split documents into chunks, embed them, index them and retrieve relevant passages. In this first conceptual lab, we use a simpler method: keyword-based classification. This lets you focus on the meaning of concepts without needing model APIs yet.

This is also why the example uses unit materials instead of a random external article. Students can inspect the source text, understand the context and later compare the rule-based approach with retrieval-based approaches in the planned RAG practicals.

<a id="m01b-core-concepts"></a>

### 3. Core Concepts

Generative AI refers to models that generate new content, such as text, images, audio or code. A large language model is a type of generative model trained to process and generate text. A chatbot is an interface pattern in which a user exchanges messages with a model. An agentic AI system goes further by adding planning, tool use, memory, retrieval, verification or action execution.

An everyday analogy helps here. A chatbot is like a knowledgeable receptionist: you ask a question, you get an answer, and nothing in the building changes. An agentic system is like an office assistant with a set of keys: it can look things up in the filing room, book meeting rooms and send letters on your behalf. The assistant is far more useful, but you now care about which keys it holds, which actions it is allowed to take, and whether someone reviews the important ones.

Before looking at workflows, it is worth having a rough picture of what happens inside an LLM when it generates text. The model does not read words the way you do. Text is first split into tokens, each token is mapped to a vector of numbers called an embedding, and transformer layers then let every token draw on the context of the whole input. The output is a probability distribution over possible next tokens; one token is chosen, appended to the text, and the process repeats until the response is complete.

```text
        "Explain agentic AI"           (input text)
                 |
                 v
        +------------------+
        | Tokenizer        |  splits text into tokens, e.g.
        |                  |  ["Explain", " agent", "ic", " AI"]
        +------------------+
                 |
                 v
        +------------------+
        | Embeddings       |  each token becomes a vector of
        |                  |  numbers the model can compute with
        +------------------+
                 |
                 v
        +------------------+
        | Transformer      |  attention layers let every token
        | layers           |  use context from the whole input
        +------------------+
                 |
                 v
        +------------------+
        | Next-token       |  e.g. " Agent": 0.31, " An": 0.12, ...
        | probabilities    |  one token is chosen and appended,
        +------------------+  then the loop repeats
```

This picture explains two behaviours you will keep meeting in this unit. First, the model only ever predicts plausible continuations, which is why fluent but unsupported answers (hallucinations) are possible. Second, everything the model knows about your task must fit into its input, which is why prompts, context and retrieval matter so much.

It is also useful to separate four layers of a complete system. The **model layer** is responsible for prediction or generation; for text, this is usually an LLM. The **prompt and context layer** determines what information the model receives at inference time. The **workflow layer** decides how the task is decomposed, whether retrieval is needed, whether tools should be called and how state is updated. The **evaluation and safety layer** checks whether the system behaves acceptably.

The difference between system types can be summarised as follows.

<div align="center">

<table>
<thead>
<tr>
<th><strong>System type</strong></th>
<th><strong>Main behaviour</strong></th>
<th><strong>Typical risk</strong></th>
<th><strong>Relevant later practical</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left">Generative AI model</td><td>Generates content from input.</td><td>May produce inaccurate or unsupported content.</td><td>M01C, M04A</td></tr>
<tr><td align="left">Chatbot</td><td>Responds conversationally to user messages.</td><td>May hallucinate or follow poor instructions.</td><td>M03B</td></tr>
<tr><td align="left">RAG assistant</td><td>Retrieves documents before answering.</td><td>May retrieve irrelevant documents or misuse retrieved evidence.</td><td>M03C, M05A, M05B</td></tr>
<tr><td align="left">Tool-using agent</td><td>Selects and calls approved tools.</td><td>May call the wrong tool or pass unsafe arguments.</td><td>M03D, M04B, M04C</td></tr>
<tr><td align="left">Stateful agentic workflow</td><td>Tracks state across multiple steps.</td><td>May accumulate incorrect state or act on stale context.</td><td>M05C</td></tr>
</tbody>
</table>

</div>

The most important transition is from response generation to action. Once a model can call tools, search the web, write files, send emails, run code or update systems, the workflow needs stronger constraints. It should define allowed tools, validate inputs, record state, test failure cases and keep a human in the loop for high-impact actions.

```text
+--------------+     +--------------------+     +------------------------+
| User request | --> | Prompt and context | --> | Model or routing logic |
+--------------+     +--------------------+     +------------------------+
                                                           |
                                                     Need a tool?
                                                     |          |
                                                     no         yes
                                                     |          |
                                                     v          v
                                     +-----------------+   +---------------------+
                                     | Generate answer |   | Validate tool input |
                                     +-----------------+   +---------------------+
                                                                      |
                                                                      v
+--------------------+     +--------------+     +--------------------+
| Review and respond | <-- | Update state | <-- | Call approved tool |
+--------------------+     +--------------+     +--------------------+
```

Read the diagram as a control-flow sketch rather than as an implementation. It shows that a responsible agentic system should not jump directly from a user request to a tool call. There are intermediate steps: interpret the request, check whether a tool is needed, validate the input, call only approved tools, and review the output before responding. You built exactly these guard steps in miniature in `M01A`, and you will build them at full scale from Module 4 onwards.

In [ ]:
def tokenize(text: str) -> List[str]:
    """Convert text into lowercase word tokens.

    This is a deliberately simple preprocessing function, not a full NLP
    tokenizer. Real LLM tokenizers split text into subword pieces and are
    trained together with the model. The purpose here is to make the idea
    of "text becomes units the machine can count" concrete before later
    sessions introduce embeddings and vector databases.
    """
    # Lowercasing first means "RAG" and "rag" count as the same token.
    # The pattern keeps words of two or more letters, including hyphenated
    # terms such as "multi-agent". Numbers and punctuation are dropped
    # because rough word counts are all this demonstration needs.
    return re.findall(r"[a-zA-Z][a-zA-Z\-]+", text.lower())


tokens = tokenize(unit_text)
counts = Counter(tokens)

print("Number of tokens:", len(tokens))
print("Most common words:")
for word, count in counts.most_common(12):
    print(f"{word:20s} {count}")

The output of the previous cell has two parts. The first line reports the number of word-like tokens extracted from the unit text. This is a rough measure of text length, not a linguistic analysis. The second part lists the most frequent words. You should expect repeated unit-specific terms such as `AI`, `agentic`, `unit`, `public`, `RAG`, `workflow` or `model` depending on the exact text.

This output demonstrates a simple but important point: before an AI system retrieves, classifies or summarises text, it needs some representation of that text. A word count is a very weak representation. Embeddings, introduced as the retrieval encoder in [M02C-Retrieval-Augmented-Generation](../../M02-Prompt-RAG/Jupyter/M02C-Retrieval-Augmented-Generation.ipynb), provide a richer representation by mapping text into vectors where semantic similarity can be measured and relevant evidence can be retrieved.

This word-count example is not generative AI by itself. It is a simple analysis step. It prepares you to understand how later systems transform text into features, embeddings, retrieved chunks or structured workflow state.

<a id="m01b-guided-implementation"></a>

### 4. Guided Implementation

We now build a small concept classifier. It does not use an LLM. It uses transparent keyword rules to classify whether a short description is mainly about generative AI, RAG, tool use, multi-agent systems, model adaptation, safety or productisation.

The classifier works like a reading assistant holding a fixed list of highlighter rules: it scans the text and highlights every phrase that appears on its list. It never understands the text; it only matches strings. This makes it a good first model to study, because every decision is visible. If the classifier labels a sentence as `rag`, you can point at the exact keyword that triggered the label. In a real LLM-based classifier, the model may infer concepts from context, which is more flexible but much harder to inspect.

The transparency also makes the limitations obvious. If a text uses a synonym that is not on the keyword list, the classifier misses the concept entirely. A sentence about "vector search over embeddings" is clearly about retrieval, but the current rules will not detect it. This blind spot is not a bug to be ashamed of; it is a property of every fixed representation, and recognising such blind spots is a skill you will reuse when evaluating far more sophisticated systems.

When you run the next cell, you should see a dictionary with `ok=True` and a `result` field mapping matched concept names to the exact keywords that triggered them. If you pass something invalid instead, such as an empty string, you should get `ok=False` with a clear error message and no crash.

In [ ]:
# Concept keywords.
# These are deliberately simple and inspectable.
# Later sessions will replace such rules with embeddings, retrieval and LLM reasoning.

CONCEPT_KEYWORDS: Dict[str, List[str]] = {
    "generative_ai": [
        "generative", "generate", "generation", "llm", "language model", "chatbot", "prompt"
    ],
    "rag": [
        "retrieval", "retrieval-augmented", "rag", "documents", "unit materials", "knowledge"
    ],
    "tool_use": [
        "tool", "tools", "function calling", "api", "action", "workflow"
    ],
    "multi_agent": [
        "multi-agent", "collaboration", "agents"
    ],
    "model_adaptation": [
        "fine-tuning", "adaptation", "model adaptation", "multimodal", "speech", "diffusion"
    ],
    "safety": [
        "safety", "safe", "prompt injection", "defence", "private", "risk"
    ],
    "productisation": [
        "productised", "go-to-market", "deployment", "hosting", "web"
    ],
}


def classify_concepts(text: str) -> Dict[str, Any]:
    """Classify a text snippet into unit concept categories.

    The function returns structured output rather than only printing labels.
    This follows the same design principle used in M01A: later workflow steps
    should be able to inspect whether the operation succeeded and what evidence
    supported the result.
    """
    if not isinstance(text, str):
        return {
            "ok": False,
            "error": "Input must be a string.",
            "result": None
        }

    cleaned = text.lower().strip()
    if not cleaned:
        return {
            "ok": False,
            "error": "Input text is empty.",
            "result": None
        }

    matches: Dict[str, List[str]] = {}

    for concept, keywords in CONCEPT_KEYWORDS.items():
        matched_keywords = []
        for keyword in keywords:
            if keyword in cleaned:
                matched_keywords.append(keyword)
        if matched_keywords:
            matches[concept] = matched_keywords

    return {
        "ok": True,
        "error": None,
        "result": matches
    }


example = "This session builds a RAG assistant over unit documents using retrieval and prompts."
classification = classify_concepts(example)
classification

The output is a dictionary with three fields.

The `ok` field tells later workflow steps whether the operation succeeded. The `error` field explains what went wrong when `ok=False`. The `result` field stores the actual concept matches. This structured-output pattern is deliberately similar to the safe tool-output pattern introduced in `M01A`.

For the example input, you should see concepts such as `rag` and possibly `generative_ai`, because the sentence contains words such as `RAG`, `retrieval`, `documents` and `prompts`. The matched keywords are shown as evidence. This is not a sophisticated explanation, but it is transparent. You can inspect exactly which words triggered the classification.

Later LLM-based systems will be more flexible, but their outputs are harder to verify. This is why the unit repeatedly asks you to produce evidence, test cases and structured outputs.

In [ ]:
def summarise_concepts(classification: Dict[str, Any]) -> str:
    """Convert structured classification output into a readable summary.

    This function is intentionally separated from classify_concepts.
    The classifier performs analysis; the summariser formats the result.
    Separating these concerns makes the workflow easier to debug.
    """
    if not classification.get("ok"):
        return f"Classification failed: {classification.get('error')}"

    result = classification.get("result", {})
    if not result:
        return "No unit concept was confidently matched."

    parts = []
    for concept, evidence in result.items():
        evidence_text = ", ".join(evidence)
        parts.append(f"{concept} evidence: {evidence_text}")

    # Join with a real newline so each concept prints on its own line.
    return "\n".join(parts)


print(summarise_concepts(classification))

The summary output is intended for a human reader. It does not replace the structured dictionary. This distinction is important in agentic workflows: one representation may be useful for machines, while another representation may be useful for people. In later sessions, a tool call, retrieval result or LangGraph state object should remain machine-readable even if the final response is written in natural language.

In [ ]:
# Apply the classifier to the public unit text.
unit_classification = classify_concepts(unit_text)
print(summarise_concepts(unit_classification))

This output should contain several matched categories because the sample unit text mentions many parts of the planned syllabus. For example, `rag` should be detected because the text mentions retrieval-augmented generation and RAG over unit materials. `tool_use` should be detected because the text mentions APIs, function calling and tools. `safety` should be detected because the text mentions prompt injection defence and private agents.

Treat this output as a concept map of the unit rather than a final classification model. The classifier is useful because it is inspectable, but it is limited because it only detects concepts that appear through known keywords. This limitation motivates later practicals on embeddings, RAG and LLM-based reasoning.

This guided implementation demonstrates three important workflow ideas: first, the system separates analysis from presentation; second, it returns structured output; third, it provides inspectable evidence. These ideas will reappear in later LLM applications, even when the internal model is more complex.

<a id="m01b-testing"></a>

### 5. Testing and Analysis

A simple classifier should be tested with normal, edge and failure cases. Normal cases check whether expected topics are detected. Edge cases check minimal but valid inputs. Failure cases check invalid inputs such as empty text or non-string objects.

Testing is especially important in agentic AI because a classification error can lead to a wrong downstream action. If a workflow misclassifies a task as tool use, it may call a tool unnecessarily. If it misses a safety topic, it may fail to apply the right guardrails.

The tests below use `assert` statements. An `assert` statement stops execution if a condition is false. If all tests pass, Python prints the final success message. If a test fails, Python raises an `AssertionError`, which indicates that either the classifier logic or the expected behaviour should be reviewed.

The test set has a deliberate structure.

<div align="center">

<table>
<thead>
<tr>
<th><strong>Test type</strong></th>
<th><strong>Purpose</strong></th>
<th><strong>Example in this notebook</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left">Normal case</td><td>Check expected behaviour on clear input.</td><td>RAG sentence detects <code>rag</code>.</td></tr>
<tr><td align="left">Edge case</td><td>Check minimal but valid input.</td><td><code>LLM prompt</code> detects <code>generative_ai</code>.</td></tr>
<tr><td align="left">Failure case</td><td>Check safe rejection of invalid input.</td><td>Empty string and list input return <code>ok=False</code>.</td></tr>
</tbody>
</table>

</div>

In [ ]:
# Normal case: a clear RAG-related statement.
normal = classify_concepts("Build a RAG assistant using retrieval over unit documents.")
assert normal["ok"] is True
assert "rag" in normal["result"]

# Normal case: a tool-use statement.
tool_case = classify_concepts("Use function calling and APIs to connect the agent to external tools.")
assert tool_case["ok"] is True
assert "tool_use" in tool_case["result"]

# Edge case: short but valid text.
edge = classify_concepts("LLM prompt")
assert edge["ok"] is True
assert "generative_ai" in edge["result"]

# Failure case: empty text.
empty = classify_concepts("   ")
assert empty["ok"] is False
assert empty["result"] is None

# Failure case: non-string input.
bad_type = classify_concepts(["RAG", "tool"])
assert bad_type["ok"] is False
assert bad_type["result"] is None

print("All concept classifier tests passed.")

If the cell prints `All concept classifier tests passed.`, each specified behaviour was satisfied. This does not mean the classifier is universally correct. It only means that it passed the behaviours we explicitly checked. This distinction matters in AI evaluation. A small test suite can provide evidence, but it cannot prove that an AI system will behave correctly on every possible input.

When you design agentic AI systems later in the unit, use this same logic: define expected behaviour, test normal cases, test boundary cases, and test safe failure. The more external action a system can take, the more important these tests become.

Before moving to the student tasks, run one more experiment of your own. Try `classify_concepts("vector search over embeddings for semantic matching")` in a new cell. The sentence is clearly about retrieval, yet the classifier will likely return no `rag` match, because none of the exact keywords appear. This is the blind spot discussed earlier, observed first-hand.

Keep this experiment in mind for your reflection. It captures the central trade-off of this session: transparent rules give you evidence you can point at, while flexible models give you coverage you cannot fully inspect. Modern AI engineering constantly balances these two.

<a id="m01b-student-tasks"></a>

### 6. Student Tasks

This section contains the required student work for this session. The earlier sections introduced the concepts and provided a guided implementation. Now you should extend the concept classifier and test its behaviour. The extension is linked to later private/local-agent practicals such as [M06C-PrivateAgents-Ollama](../../M06-Multi-Agent-Safety/Jupyter/M06C-PrivateAgents-Ollama.ipynb).

<div align="center">

<table>
<thead>
<tr>
<th><strong>Task</strong></th>
<th><strong>What you need to do</strong></th>
<th><strong>Why it matters</strong></th>
<th><strong>Expected evidence</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left">Task 1</td><td>Add a new concept category called <code>open_source_agents</code>.</td><td>This connects the classifier to later sessions on private and local agents.</td><td>The keyword dictionary includes the new category.</td></tr>
<tr><td align="left">Task 2</td><td>Add at least five keywords for the new category.</td><td>Keyword coverage affects what the classifier can detect.</td><td>The new category detects terms such as <code>Ollama</code>, <code>local LLM</code> or <code>open-source</code>.</td></tr>
<tr><td align="left">Task 3</td><td>Run normal, edge and failure tests.</td><td>Testing shows whether the new category works without breaking existing behaviour.</td><td>All tests pass and at least one test detects <code>open_source_agents</code>.</td></tr>
<tr><td align="left">Task 4</td><td>Write a short explanation of how this classifier differs from an LLM-based classifier.</td><td>This helps distinguish transparent rules from model-based reasoning.</td><td>A short explanation in your reflection.</td></tr>
</tbody>
</table>

</div>

Expected behaviour for the new category:

<div align="center">

<table>
<thead>
<tr>
<th><strong>Input case</strong></th>
<th><strong>Example</strong></th>
<th><strong>Expected behaviour</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left">Normal case</td><td><code>Build a private agent with Ollama and a local LLM.</code></td><td>Detect <code>open_source_agents</code>.</td></tr>
<tr><td align="left">Edge case</td><td><code>Ollama</code></td><td>Detect <code>open_source_agents</code> from a minimal valid input.</td></tr>
<tr><td align="left">Failure case</td><td><code>""</code></td><td>Return <code>ok=False</code> and a clear error message.</td></tr>
<tr><td align="left">Failure case</td><td><code>None</code></td><td>Return <code>ok=False</code> because the input is not a string.</td></tr>
</tbody>
</table>

</div>

In [ ]:
# Student task starter.
#
# Goal: teach the classifier to recognise open-source and private-agent topics.
#
# Step 1: use the exact category name "open_source_agents".
#         The test cell below looks for this exact string, so a different
#         name (or a typo) will make the tests fail even if your logic works.
#
# Step 2: assign a list of at least five keywords, all in lowercase.
#         Matching is a simple substring check against lowercased input text,
#         so "ollama" will match "Ollama" in a sentence, but "Ollama" as a
#         keyword would never match anything.
#
# Suggested keywords:
#   "ollama", "local llm", "open-source", "private agent", "llama", "mistral"
#
# TODO: add your new category by uncommenting and completing the line below.
# CONCEPT_KEYWORDS["open_source_agents"] = [...]

# After editing, re-run this cell so the dictionary is updated, then run the
# test cell below. If you re-run the cell that defines CONCEPT_KEYWORDS
# (in Section 4), your category is erased and must be added again.
print("Categories currently known:", list(CONCEPT_KEYWORDS.keys()))

In [ ]:
# Student task tests.
# Uncomment and run after adding the new category.

# normal_open_source = classify_concepts("Build a private agent with Ollama and a local LLM.")
# assert normal_open_source["ok"] is True
# assert "open_source_agents" in normal_open_source["result"]

# edge_open_source = classify_concepts("Ollama")
# assert edge_open_source["ok"] is True
# assert "open_source_agents" in edge_open_source["result"]

# failure_empty = classify_concepts("")
# assert failure_empty["ok"] is False

# failure_none = classify_concepts(None)
# assert failure_none["ok"] is False

# print("Student task tests passed.")

<a id="m01b-submission"></a>

### 7. Submission and Reflection

Submit the completed notebook with the following evidence.

<div align="center">

<table>
<thead>
<tr>
<th><strong>Required item</strong></th>
<th><strong>What to submit</strong></th>
<th><strong>Quality check</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left">Completed classifier</td><td>The updated <code>CONCEPT_KEYWORDS</code> dictionary with <code>open_source_agents</code>.</td><td>The new category should contain at least five relevant lowercase keywords.</td></tr>
<tr><td align="left">Tests</td><td>Normal, edge and failure tests for the new category.</td><td>Tests should run without unexpected errors.</td></tr>
<tr><td align="left">Short analysis</td><td>A brief comparison between rule-based and LLM-based concept classification.</td><td>The explanation should mention transparency, flexibility and limitations.</td></tr>
<tr><td align="left">Reflection</td><td>150 to 250 words.</td><td>The reflection should connect this session to later agentic AI workflows.</td></tr>
</tbody>
</table>

</div>

Reflection questions:

1. What is the difference between a chatbot and an agentic workflow?
2. Why does tool use increase the need for validation and testing?
3. What are the strengths and weaknesses of the rule-based classifier used in this lab?
4. How would an LLM-based classifier be more flexible?
5. What new risks appear when an AI system can retrieve documents or call tools?

Use the debugging guide below if your notebook does not behave as expected.

<div align="center">

<table>
<thead>
<tr>
<th><strong>Symptom</strong></th>
<th><strong>Likely cause</strong></th>
<th><strong>How to inspect</strong></th>
<th><strong>Typical fix</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left">New category never detected</td><td>Keywords contain uppercase letters or the starter cell was not re-run</td><td>Print <code>CONCEPT_KEYWORDS["open_source_agents"]</code> and check the values</td><td>Use lowercase keywords and re-run the starter cell</td></tr>
<tr><td align="left"><code>KeyError: open_source_agents</code></td><td>The Section 4 cell defining <code>CONCEPT_KEYWORDS</code> was re-run after the starter cell, erasing your category</td><td>Print <code>list(CONCEPT_KEYWORDS.keys())</code></td><td>Re-run the starter cell so the category is added again</td></tr>
<tr><td align="left"><code>NameError: classify_concepts is not defined</code></td><td>Cells were run out of order</td><td>Check which cells show an execution number</td><td>Run all cells from the top with <em>Runtime → Run all</em></td></tr>
<tr><td align="left"><code>AssertionError</code> in a test</td><td>Expected keyword missing from the category or wrong category name</td><td>Call <code>classify_concepts</code> manually on the failing sentence and inspect <code>result</code></td><td>Add the missing keyword or fix the category name</td></tr>
<tr><td align="left">Empty input not rejected</td><td>Validation logic in <code>classify_concepts</code> was changed</td><td>Call <code>classify_concepts("")</code> and inspect the output</td><td>Restore the empty-string check before keyword matching</td></tr>
</tbody>
</table>

</div>

#### Further Readings

- Google AI for Developers: <https://ai.google.dev>
- OpenAI Platform documentation: <https://platform.openai.com/docs>
- LangChain documentation: <https://python.langchain.com/docs/introduction/>
- LangGraph documentation: <https://langchain-ai.github.io/langgraph/>
- Flowise documentation: <https://docs.flowiseai.com>
- Hugging Face unit: <https://huggingface.co/learn>